# 2a · Where are the policy-relevant tipping points?

**Research question (RQ 2a):** *The IQB defines both "minimum quality" and "high quality" thresholds for each use case. At which percentile does a country flip from meeting minimum requirements to falling short? If small threshold adjustments push countries across classification boundaries, then policy decisions built on these scores may be implicated.*

We answer this in four steps.

**Step 1: Find each use case's tipping percentile.** A use case meets its thresholds at percentile *q* when every one of its requirements passes. A requirement passes at *q* exactly when its pass rate is above 1 − *q* (see *Key idea: pass rates* in the README), so the use case meets its thresholds when

$$q > \text{tipping percentile} = 100 \times \big(1 - \min_{\text{requirements}} \text{pass rate}\big).$$

We compute this for the framework's **minimum-quality** and **high-quality** thresholds (`FRAMEWORK_MIN` and `FRAMEWORK_HIGH` in `iqb_tools.py`, from the IQB framework table).

**Step 2: Classify each country at a percentile.** We use four ordered classes:
- *High quality*: all six use cases meet their high-quality thresholds
- *Minimum quality*: all six meet their minimum thresholds
- *Partial*: at least one use case meets its minimum, but not all
- *Below minimum*: no use case meets its minimum

**Step 3: Count how many countries flip** between percentile choices (P25, P50, P75, P90, P95), how many sit within ±5 percentile points of a boundary, and how many change class under small threshold adjustments (±10%, ±25%) or with the library's thresholds.

**Step 4: Find which requirement decides** each country's tipping point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import iqb_tools as T

T.set_style()
cfg = T.config_table()  # IQB library thresholds
f_min = T.with_thresholds(cfg, T.FRAMEWORK_MIN)  # framework, minimum quality
f_high = T.with_thresholds(cfg, T.FRAMEWORK_HIGH)  # framework, high quality

# Sample: at least the minimum number of tests in 2025 (1a) and a World Bank income group
ctx = T.read_table("country_context").set_index("country_code")  # written by 0-setup
d = ctx[(T.binding_tests(ctx) >= T.MIN_TESTS) & ctx.income_group.notna()].copy()
d["income"] = pd.Categorical(d.income_group, T.INCOME_ORDER, ordered=True)
print(len(d), "countries")

# 2025 pass rates per country: the test-weighted average of the 12 months
df25 = T.load_level("country", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS"))
df25 = df25[df25.country_code.isin(d.index)]


def pooled(tbls):
    """2025 pass rates of every test used by the given threshold tables."""
    tests = T.distinct_tests(*tbls)
    pr = T.pass_rates(df25, tests)
    return T.pool_pass_rates(df25, pr, "country_code", tests).loc[d.index]


C = pooled([cfg, f_min, f_high])

TIERS = ["Below minimum", "Partial", "Minimum quality", "High quality"]
TIER_COLORS = [T.BLUE_RAMP[k] for k in (250, 400, 550, 700)]  # ordered classes -> one-hue ramp


def tipping(pr, tbl_min, tbl_high):
    """Per-use-case tipping percentiles and the three country-level boundaries."""
    tmin = pd.DataFrame({u: T.tipping_percentile(pr, tbl_min, u) for u in T.USE_CASES})
    thigh = pd.DataFrame({u: T.tipping_percentile(pr, tbl_high, u) for u in T.USE_CASES})
    bounds = pd.DataFrame(
        {
            "any_min": tmin.min(axis=1),  # enters "Partial"
            "all_min": tmin.max(axis=1),  # enters "Minimum quality"
            "all_high": thigh.max(axis=1),  # enters "High quality"
        }
    )
    return tmin, thigh, bounds


def classify(bounds, q):
    return pd.Series(
        np.select([q > bounds.all_high, q > bounds.all_min, q > bounds.any_min], [3, 2, 1], 0),
        index=bounds.index,
    )


tmin, thigh, B = tipping(C, f_min, f_high)
B.groupby(d.income, observed=True).median().round(1).rename(
    columns={"any_min": "enters Partial above", "all_min": "enters Minimum above", "all_high": "enters High above"}
)

The table reports, for the median country of each income group, the percentile above which it enters each class. 
Each country has three such tipping points, one for each class boundary, computed as in Step 1 from its 2025 pass rates:

**enters Partial above**:The lowest of the country's six minimum-quality tipping percentiles: the point where its first use case meets the minimum thresholds. Below it, the country is *Below minimum*.

**enters Minimum above**: The highest of the six: the point where its last use case meets the minimum, so that all six do. 

**enters High above**: The highest of the six high-quality tipping percentiles: the point where all six use cases meet their high-quality thresholds. 

The rows are World Bank income groups, covering the 201 countries with at least 1,153 tests in 2025 and an income group. Each column's median is taken separately, so the three numbers in a row need not come from the same country. A lower number is better: the country reaches the class at a weaker part of its test distribution.

For example, the median low-income country meets the minimum thresholds of all six use cases only when scored above about P98, while the median high-income country does so above about P59. Read against the usual percentile choices, the median country at P50 is *Below minimum* in the three lower income groups and *Partial* in the high-income group. At P95 it is *Partial* in the low- and lower-middle-income groups and *Minimum quality* in the two upper groups, with the high-income median (95.4) just short of *High quality*.


In [ ]:
# Worked example: Brazil's tipping percentiles by use case, and the three boundaries they give
display(pd.DataFrame({"minimum quality": tmin.loc["BR"], "high quality": thigh.loc["BR"]}).T.round(1))
br = B.loc["BR"]
print(f"Brazil enters Partial above P{br.any_min:.1f}, Minimum quality above P{br.all_min:.1f}, "
      f"High quality above P{br.all_high:.1f}")

# A tipping point of 100 means never: some requirement has an estimated pass rate of 0
never = B.all_high >= 100
up200 = C[T.test_key("upload", 200)] <= 0  # online backup's high-quality upload threshold
print(f"\nNever reach High quality: {never.sum()} of {len(B)} countries;",
      never.groupby(d.income, observed=True).sum().to_dict())
print(f"of these, upload above 200 Mbit/s has a pass rate of 0 in {(never & up200).sum()}:",
      (never & up200).groupby(d.income, observed=True).sum().to_dict())


Brazil shows how the three numbers arise. Its minimum-quality tipping percentiles are 40.4 for web browsing, video streaming, audio streaming and gaming, 45.2 for online backup and 47.0 for video conferencing. So Brazil enters *Partial* above P40.4, when its first use cases meet the minimum, and *Minimum quality* above P47.0, when the last one does. Its highest high-quality tipping percentile is 89.2, from online backup, so it enters *High quality* above P89.2. Scored at P50, Brazil is *Minimum quality*; scored at P95, it is *High quality*.

A tipping point of 100 means the class is never reached: at least one requirement has an estimated pass rate of zero, so no percentile meets it. For *High quality* this is the case in 104 of the 201 countries, including every low-income country and 38 of the 43 lower-middle-income ones, which is why those two medians are 100.0. In 89 of the 104, the cause is online backup's high-quality requirement of 200 Mbit/s upload: the country's reported P99 upload speed is below 200 Mbit/s.

## Where does each country tip?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
rng = np.random.default_rng(5)  # vertical jitter only
for ax, (col, title) in zip(
    axes,
    [
        ("all_min", "Meets minimum quality for all six use cases above …"),
        ("all_high", "Meets high quality for all six use cases above …"),
    ],
):
    for i, inc in enumerate(T.INCOME_ORDER):
        x = B.loc[d.income == inc, col]
        ax.scatter(x, i + rng.uniform(-0.2, 0.2, len(x)), s=18, color=T.INCOME_COLORS[inc],
                   edgecolor=T.SURFACE, linewidth=0.9)
    for p in (25, 50, 75, 95):
        ax.axvspan(p - 5, p + 5, color=T.GRID, alpha=0.6, lw=0, zorder=0)
        ax.text(p, 3.55, f"P{p} ± 5", ha="center", color=T.MUTED, fontsize=9)
    ax.set_xlim(0, 101)
    ax.set_ylim(-0.5, 3.8)
    ax.set_title(title)
    ax.set_xlabel("tipping percentile")
axes[0].set_yticks(range(4), [T.INCOME_SHORT[i] for i in T.INCOME_ORDER])
fig.suptitle("Country tipping points (dots: countries; bands: ±5 points around common percentile choices)",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
T.savefig(fig, "2a_tipping_points_by_income")

Each dot is one country, placed at its tipping percentile. In the left panel, that is the percentile above which the country meets the minimum thresholds of all six use cases (*enters Minimum above* in the table); in the right panel, the percentile above which it meets all six high-quality thresholds (*enters High above*). The x-axis runs from P0 to P100. The y-axis groups the countries by income; within a row, the dots are spread up and down at random only so that they don't overlap, so their vertical position carries no meaning. The gray bands mark ±5 points around P25, P50, P75 and P95, and a dot at 100 never reaches the class.

To read a dot, compare its position with the percentile used for scoring: the country is in the class when scored at any percentile to the right of its dot. Brazil's dot in the left panel sits at 47.0, so Brazil meets the minimum thresholds of all six use cases when scored at P50, but not at P25. A dot inside a gray band belongs to a country whose class at that percentile would change if the percentile moved by less than five points.

The dots move right as income falls. In the left panel, high-income countries spread from P28 (Monaco, the only dot in the P25 band) to P100, and about half of them tip below P59. The lowest dot is at P47 for upper-middle-income countries, P67 for lower-middle-income ones and about P80 for low-income ones, and half or more of the dots in these three groups lie at P90 or above (20 of 22 low-income, 28 of 43 lower-middle-income and 27 of 54 upper-middle-income countries). In the right panel, only one country (Monaco again, at P59) tips below P70, and only six tip below P80. Every low-income country and 38 of the 43 lower-middle-income ones sit at 100: they never reach *High quality*. In both panels the P95 band is the most crowded, which is why so many countries are on the edge at the framework default (see below).

## How does the percentile change a country's class?

In [ ]:
LEVELS = [25, 50, 75, 90, 95]
cls = pd.DataFrame({q: classify(B, q) for q in LEVELS})
comp = pd.DataFrame(
    {f"P{q}": cls[q].value_counts().reindex(range(4), fill_value=0).to_numpy() for q in LEVELS},
    index=TIERS,
)
comp

The table counts the countries in each class (rows) when all of them are scored at the same percentile (columns), so each column adds up to 201. A higher percentile passes at least as many requirements as a lower one, so a country's class can only stay the same or improve as the percentile rises: reading from left to right, countries move down the table. *Below minimum* shrinks from 198 countries at P25 to 20 at P95, while *High quality* is empty until P75 (2 countries) and holds 48 at P95.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
for j, q in enumerate(LEVELS):
    left = 0
    for k, tier in enumerate(TIERS):
        v = comp.loc[tier, f"P{q}"] / len(d)
        if v > 0:
            ax.barh(j, v - 0.004, left=left + 0.002, color=TIER_COLORS[k], height=0.5,
                    label=tier if tier not in ax.get_legend_handles_labels()[1] else None)
        if v > 0.06:
            ax.text(left + v / 2, j, f"{v:.0%}", ha="center", va="center", fontsize=9,
                    color="white" if k >= 2 else T.INK)
        left += v
ax.set_yticks(range(len(LEVELS)), [f"P{q}" for q in LEVELS])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="y", visible=False)
ax.set_title(f"Class of each country under different percentile choices ({len(d)} countries)")
handles, labels = ax.get_legend_handles_labels()
order = [labels.index(t) for t in TIERS if t in labels]
ax.legend([handles[i] for i in order], [labels[i] for i in order], ncol=4, loc="upper center",
          bbox_to_anchor=(0.5, -0.15))
T.savefig(fig, "2a_classes_by_percentile")

The figure shows the same counts as shares. Each bar is one percentile choice, listed on the y-axis from P25 at the top to P95 at the bottom; the x-axis is the share of the 201 countries, from 0% to 100%. Each bar is split into the four classes, from *Below minimum* (lightest) on the left to *High quality* (darkest) on the right, and segments larger than 6% are labeled.

Moving down the bars, the light segment shrinks and the dark ones grow. *Below minimum* falls from 99% of countries at P25 to 63% at P50, 33% at P75, 18% at P90 and 10% at P95, so the largest shifts happen between P25 and P75. *Minimum quality* is the largest class from P75 on (42% at P75 and P90, 39% at P95). *High quality* first appears as a sliver at P75 (1%) and reaches 13% at P90 and 24% at P95.

In [ ]:
# Transitions between the "typical test" view (P50) and the framework default (P95)
pd.crosstab(
    cls[50].map(dict(enumerate(TIERS))),
    cls[95].map(dict(enumerate(TIERS))),
    rownames=["class at P50"],
    colnames=["class at P95"],
).reindex(index=TIERS, columns=TIERS, fill_value=0)

Each country is counted once, in the row of its class at P50 and the column of its class at P95. The diagonal (20, 2, 2 and 0) holds the 24 countries whose class is the same at both percentiles; the other 177 change class. All of them are to the right of the diagonal, because no country can do worse at P95 than at P50. For example, 53 countries are *Below minimum* at P50 and *Partial* at P95, and 6 jump from *Below minimum* straight to *High quality*. The bottom row is empty because no country reaches *High quality* at P50.

In [ ]:
changed = cls[50] != cls[95]
by_inc = pd.DataFrame(
    {
        "countries": d.income.value_counts(sort=False),
        "class differs P25 vs P50": (cls[25] != cls[50]).groupby(d.income, observed=True).mean(),
        "class differs P50 vs P95": changed.groupby(d.income, observed=True).mean(),
        "class differs P75 vs P95": (cls[75] != cls[95]).groupby(d.income, observed=True).mean(),
        "below minimum at P25": (cls[25] == 0).groupby(d.income, observed=True).mean(),
        "below minimum at P50": (cls[50] == 0).groupby(d.income, observed=True).mean(),
        "below minimum at P95": (cls[95] == 0).groupby(d.income, observed=True).mean(),
    }
)
print(f"class differs between P50 and P95 for {changed.sum()} of {len(d)} countries ({changed.mean():.0%})")
by_inc.round(2)

The printed line counts the countries whose class at P50 differs from their class at P95. The table breaks this down by income group:

| Column | Meaning |
|---|---|
| *countries* | The number of countries in the income group. |
| *class differs P25 vs P50* | The share of the group whose class at P25 differs from their class at P50 (0.67 means 67%). |
| *class differs P50 vs P95*, *class differs P75 vs P95* | The same for the other two pairs of percentiles. |
| *below minimum at P25*, *P50*, *P95* | The share of the group that is *Below minimum*, meeting no use case's minimum thresholds, at that percentile. |

The share whose class differs between P25 and P50 rises steeply with income, from 0% of low-income countries to 67% of high-income ones. For low-income countries this only means that they are *Below minimum* at both percentiles. Between P50 and P95, most countries in every group change class (77–94%). The *below minimum* shares fall with income at P50 and P95: at P50 they are 100% for low-income, 93% for lower-middle-income, 72% for upper-middle-income and 32% for high-income countries.

In [ ]:
pd.crosstab(d.income, cls[95].map(dict(enumerate(TIERS))), normalize="index").reindex(columns=TIERS).rename(
    index=T.INCOME_SHORT
).round(2).rename_axis(columns="class at P95")

Each row is an income group, and the values are the shares of its countries in each class at P95, so each row adds up to 1 apart from rounding. For example, 0.49 in the last column means that 49% of high-income countries are *High quality* at P95. Moving down the rows, the shares shift to the right: *High quality* goes from 0% of low- and lower-middle-income countries to 15% of upper-middle-income and 49% of high-income ones. The most common class at P95 is *Partial* for low-income countries (59%), *Minimum quality* for the two middle groups (49% and 46%) and *High quality* for high-income countries (49%).

Taken together, these tables and the figure show that the percentile choice decides most countries' classification. At P25 (a weak connection), 198 of the 201 countries meet no use case's minimum thresholds. At P50 (the typical test), about two thirds still meet none. At P95 (the framework default), only about one in ten fail to, and a quarter meet the high-quality thresholds of every use case.

Nearly nine in ten countries change class between P50 and P95, and six go straight from *below minimum* at P50 to *high quality* at P95. Every low-income and nearly every lower-middle-income country in the sample is *below minimum* at P50; at P95 most of them are *partial* or *minimum quality*. Whether a funding or eligibility rule built on IQB treats these countries as served or underserved is therefore decided by the percentile choice, not by their networks.

## How many countries sit on a boundary?

We call a country **on the edge** at a percentile *q* if one of its class boundaries lies within ±5 points of *q*. A small change in the percentile, or the month-to-month movement seen in 1a, can then change its class.

In [ ]:
def on_edge(q, width=5):
    return ((B - q).abs() < width).any(axis=1)


edge = pd.DataFrame({f"P{q}": on_edge(q) for q in LEVELS})
print("share of countries with a class boundary within ±5 percentile points:")
pd.DataFrame(
    {"all countries": edge.mean(), **{T.INCOME_SHORT[i]: edge[d.income == i].mean() for i in T.INCOME_ORDER}}
).round(2)

Each value is the share of countries with at least one of their three tipping points (*enters Partial*, *enters Minimum* or *enters High*) less than five points from the percentile in the row: the share of countries **on the edge** at that percentile. The first column covers all 201 countries and the others one income group each; 0.74 means 74%. For example, at P50, 35% of upper-middle-income countries have a tipping point between P45 and P55, so scoring them at P45 or P55 instead of P50 could change their class.

The share on the edge is highest at the top percentiles: 4% of all countries at P25, 20–25% at P50 and P75, 41% at P90 and 74% at P95. The jump is largest for low-income countries: none is on the edge at P25 or P50, but 91% are at P95.

In [ ]:
# Do edge countries also have uncertain scores? Confidence that the 2025 P95 score is within ±0.05 (as in 1b)
conf95 = pd.Series(
    {cc: T.score_uncertainty(C.loc[cc], C.loc[cc, "n_down"], (95,), key=f"{cc}|2025").loc[95, "conf"] for cc in d.index}
)
confident = conf95 >= T.CONF
print(
    f"At P95: {edge['P95'].sum()} edge countries. Share whose 2025 P95 score is confident "
    f"(±{T.TOL} at {T.CONF:.0%}): edge {confident[edge['P95']].mean():.0%}, "
    f"non-edge {confident[~edge['P95']].mean():.0%}"
)

At the framework default, about three quarters of countries are within five percentile points of a class boundary, and nine in ten low-income countries are. The reason is that at P95 a boundary moves whenever a requirement's pass rate is anywhere between 0 and 10%, and for many use cases, especially the high-quality ones, that is where pass rates sit. At P25 almost no country is on the edge, but only because almost every country is far below every boundary.

The printed line asks whether these countries' scores are also uncertain. It compares the 148 countries on the edge at P95 with the other 53, by the share whose 2025 P95 score is within ±0.05 of the true score with 95% confidence (as in 1b). Most edge countries (93%) still have a confident P95 *score*. Their *classification* is fragile because of the percentile choice, not because of sampling, so a class assignment for these countries should be published with a flag, or as a range across P90–P99.

## What if the thresholds move a little?

We recompute the 2025 pass rates with every threshold made stricter or more lenient by 10% and 25% (throughput multiplied by the factor; latency and loss divided by it), and with the library's thresholds in place of the framework minimum. We then count the countries whose class changes at P50 and at P95.

In [ ]:
scenarios = {}
for f in (0.75, 0.9, 1.1, 1.25):
    lo, hi = T.scale_thresholds(f_min, f), T.scale_thresholds(f_high, f)
    label = f"{'stricter' if f > 1 else 'more lenient'} by {abs(f - 1):.0%}"
    scenarios[label] = (pooled([lo, hi]), lo, hi)
lib_min = cfg  # library thresholds as the "minimum" set
scenarios["library thresholds as minimum"] = (pooled([lib_min, f_high]), lib_min, f_high)

rows = []
for name, (pr, lo, hi) in scenarios.items():
    Bs = tipping(pr, lo, hi)[2]
    for q in (50, 95):
        new = classify(Bs, q)
        rows.append((name, f"P{q}", (new != cls[q]).sum(), (new > cls[q]).sum(), (new < cls[q]).sum()))
pert = pd.DataFrame(rows, columns=["scenario", "percentile", "countries changing class", "up", "down"])
pert.set_index(["scenario", "percentile"])[["countries changing class", "up", "down"]].unstack("percentile").loc[
    list(scenarios)
]

Each row is one way of changing the thresholds, compared with the framework thresholds used so far. *More lenient by 25%* multiplies every throughput threshold by 0.75 (10 Mbit/s becomes 7.5 Mbit/s) and divides every latency and loss threshold by 0.75 (100 ms becomes about 133 ms). *Stricter by 25%* multiplies throughput thresholds by 1.25 and divides latency and loss thresholds by 1.25 (12.5 Mbit/s and 80 ms), and the 10% scenarios work the same way. *Library thresholds as minimum* replaces the framework's minimum thresholds with the IQB library's. The columns come in three pairs, each for countries scored at P50 and at P95: *countries changing class* is the number whose class differs from the one they had with the framework thresholds, and *up* and *down* split them into those moving to a better class and those moving to a worse one.

As expected, lenient thresholds only move countries up and stricter ones only move them down. A 25% change moves between about two and three times as many countries as a 10% change, and every scenario moves more countries at P95 than at P50. The library thresholds are the only scenario that moves countries in both directions (6 up and 18 down at P50); at P95, all 46 of its changes are down.

In [ ]:
# Why do countries drop under the library thresholds at P95?
pr_lib = scenarios["library thresholds as minimum"][0]
tmin_lib = tipping(pr_lib, lib_min, f_high)[0]
drops = classify(tipping(pr_lib, lib_min, f_high)[2], 95) < cls[95]
print(f"{drops.sum()} countries drop a class at P95. Use case that now sets their minimum-quality boundary:")
print(tmin_lib[drops].idxmax(axis=1).value_counts().to_string())
print(
    f"Gaming's 10 ms latency threshold fails at P95 (pass rate ≤ 5%) for "
    f"{(pr_lib.loc[drops, 'latency<10'] <= 0.05).sum()} of them"
)

The printout follows the 46 countries that drop a class at P95 under the library thresholds. It names the use case that now sets their *Minimum quality* boundary, the one with the highest tipping percentile and so the last to meet its minimum, and counts how many of them fail gaming's 10 ms latency requirement at P95, that is, have 5% or fewer of their tests below 10 ms.

In [ ]:
plot = pert.pivot(index="scenario", columns="percentile", values="countries changing class").loc[list(scenarios)]
fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = np.arange(len(plot))
for k, (q, c) in enumerate(zip(["P50", "P95"], [T.SERIES[0], T.SERIES[1]])):
    yy = y + (k - 0.5) * 0.32
    ax.barh(yy, plot[q] / len(d), height=0.3, color=c, label=f"scored at {q}")
    for yi, n in zip(yy, plot[q]):
        ax.text(n / len(d) + 0.004, yi, f"{n}", va="center", fontsize=8, color=T.INK_2)
ax.set_yticks(y, plot.index)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="y", visible=False)
ax.set_xlabel("share of countries whose class changes (labels: number of countries)")
ax.set_title("Sensitivity of the classification to threshold changes")
ax.set_xlim(0, plot.to_numpy().max() / len(d) * 1.12)  # room for the labels
ax.legend(ncol=2, loc="lower right", bbox_to_anchor=(1.0, 1.0))
T.savefig(fig, "2a_threshold_perturbation")

The figure shows the *countries changing class* columns of the table. Each pair of bars is one scenario (y-axis). The x-axis is the share of the 201 countries whose class changes, and the label at the end of each bar gives the number of countries. Blue bars are for countries scored at P50, orange bars for P95. The figure shows how many countries change class but not in which direction; the *up* and *down* columns of the table give that.

The orange bar is longer than the blue one in every scenario, so the classification at the framework default is more sensitive to the thresholds than the classification of the typical test. The bars grow with the size of the change, and at P95, replacing the minimum thresholds with the library's moves as many countries (46, or 23%) as relaxing every threshold by 25%.

A 10% change in every threshold moves 6–9% of countries across a class boundary, and a 25% change moves 12–23%. Replacing the framework's minimum thresholds with the library's (stricter for gaming and video conferencing, more lenient for audio-streaming upload) moves 24 countries at P50 (18 down, 6 up) and 46, almost a quarter, at P95, all of them down. Every one of the 46 drops because of gaming: the library's 10 ms latency threshold is not met even by the best 5% of their tests. The thresholds are expert judgments, and their exact values matter as much for classification as the percentile.

## Which requirement decides the tipping point?

In [ ]:
def binding(pr, tbl_min, tmin):
    """Use case and test that set the 'all minimum' boundary."""
    uc = tmin.idxmax(axis=1)
    out = []
    for cc, u in uc.items():
        g = tbl_min[tbl_min.use_case == u]
        keys = [T.test_key(m, th) for m, th in zip(g.metric, g.threshold)]
        out.append((cc, u, pr.loc[cc, keys].astype(float).idxmin()))
    return pd.DataFrame(out, columns=["country_code", "use_case", "test"]).set_index("country_code")


bind = binding(C, f_min, tmin).join(d.income)
print("Use case that sets the minimum-quality boundary:")
print(bind.use_case.value_counts().to_string())
pd.crosstab(bind.test, bind.income).reindex(columns=T.INCOME_ORDER).rename(columns=T.INCOME_SHORT)

The printout names the use case that sets each country's *Minimum quality* boundary: the one with the highest tipping percentile, which is the last to meet its minimum thresholds. The table goes one step further: within that use case, it finds the requirement with the lowest pass rate, which is the one that decides the tipping point, and counts the countries for each such requirement (rows) by income group (columns). Each column therefore adds up to the size of the group (22, 43, 54 and 82). The row labels name the tests: *upload>25* is upload above 25 Mbit/s, *latency<50* is latency below 50 ms, *loss<0.005* is packet loss below 0.5%, and so on.

Upload above 25 Mbit/s decides most often in low-income (14 of 22) and high-income countries (51 of 82). In the two middle groups, latency below 50 ms is slightly more common (22 against 19 lower-middle-income countries, and 25 against 24 upper-middle-income ones). Packet loss decides only in upper-middle-income (4) and high-income countries (13).

In [ ]:
# Per use case: share of countries meeting the minimum at P50 and P95, by income group
rows = []
for u in T.USE_CASES:
    for q in (50, 95):
        meets = tmin[u] < q
        rows.append({"use case": u, "percentile": f"P{q}",
                     **{T.INCOME_SHORT[i]: meets[d.income == i].mean() for i in T.INCOME_ORDER}})
uc_tbl = pd.DataFrame(rows).set_index(["use case", "percentile"])
uc_tbl.round(2)

Each row is a use case scored at P50 or P95, and each value is the share of countries in the income group (column) that meet all of that use case's minimum thresholds at that percentile. For example, 0.68 in the first row means that 68% of high-income countries meet the web-browsing minimum when scored at P50.

Shares never fall as income rises, and they are much higher at P95 than at P50: the web-browsing minimum is met at P50 by 0% of low-income and 68% of high-income countries, and at P95 by 77% and 96%. No low-income country meets any use case's minimum at P50. Video conferencing is the hardest use case and online backup the next hardest: in every income group, their shares are the lowest (or tied for lowest).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))
seq = plt.matplotlib.colors.LinearSegmentedColormap.from_list("seq", [T.BLUE_RAMP[100], T.BLUE_RAMP[700]])
im = ax.imshow(uc_tbl.to_numpy(), cmap=seq, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(4), uc_tbl.columns)
ax.set_yticks(range(len(uc_tbl)), [f"{u} · {q}" for u, q in uc_tbl.index])
ax.grid(False)
ax.set_title("Share of countries meeting the use case's minimum thresholds")
ax.set_xlabel("income group (values in the table above)")
fig.colorbar(im, ax=ax, fraction=0.04, format=plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
T.savefig(fig, "2a_use_case_minimum_by_income")

The heatmap shows the table above as colors. The x-axis is the income group, and the y-axis lists each use case twice, scored at P50 and then at P95. A cell's color gives the share of the group's countries that meet the use case's minimum thresholds, from 0% (lightest) to 100% (darkest), as on the color bar to the right.

Colors darken from left to right, as income rises, and from each P50 row to the P95 row below it. The P50 rows are almost white for low- and lower-middle-income countries. At P95, video conferencing is the lightest row in every income group, and online backup the next lightest (tied with it for low-income countries).

In [ ]:
# Web browsing at P50 in low- and lower-middle-income countries
wb_ok = tmin["web browsing"] < 50
for inc in ["Low income", "Lower middle income"]:
    names = list(d.loc[wb_ok & (d.income == inc), "country"])
    print(f"{inc}: {len(names)} meet the web-browsing minimum at P50 {names}")
g = f_min[f_min.use_case == "web browsing"]
keys = [T.test_key(m, th) for m, th in zip(g.metric, g.threshold)]
failing = C.loc[d.income.isin(["Low income", "Lower middle income"]) & ~wb_ok, keys]
print(f"Of the {len(failing)} that do not, the requirement failing at P50 (pass rate ≤ 50%):")
print((failing <= 0.5).sum().sort_values(ascending=False).to_string())

Video conferencing sets the boundary for almost every country (197 of 201). It has the strictest minimum requirements (50 ms latency, 0.5% loss, 25 Mbit/s upload), so it is the last use case a country satisfies. Within it, the binding test is most often **upload above 25 Mbit/s** (108 countries), then **latency below 50 ms** (72) and **packet loss below 0.5%** (17, mostly high-income). Upload capacity, not download speed, is what keeps most countries from meeting the minimum for every use case.

The use-case table also shows how demanding the minimum thresholds are for the typical test. At P50, *no* low-income country and only three lower-middle-income countries (Laos, Lesotho, Venezuela) meet even the web-browsing minimum (10 Mbit/s each way, 100 ms, 1% loss). In the others the median test fails several of the four requirements, most often upload (54 of 62) and packet loss (50), then download (44) and latency (32). A country is counted once for each requirement it fails (pass rate at or below 50%), so these counts add up to more than 62. Web browsing, audio streaming and gaming have identical minimum thresholds in the framework table, which is why their rows are identical.

Where packet loss binds, a measurement caveat applies. NDT's loss figure comes from TCP retransmissions during a single bulk transfer, which are non-zero even on healthy networks when the transfer fills the path. A 0.5% threshold on that measure is strict, and how it relates to the loss that applications actually experience should be validated before it decides a classification.

## Save

In [ ]:
out = (
    d[["country", "income_group", "region"]]
    .join(B.round(2))
    .join(tmin.add_prefix("tip_min_").round(1))
    .join(thigh.add_prefix("tip_high_").round(1))
)
for q in LEVELS:
    out[f"class_p{q}"] = cls[q].map(dict(enumerate(TIERS)))
    out[f"edge_p{q}"] = on_edge(q)
out["binding_use_case"], out["binding_test"] = bind.use_case, bind.test
out.to_csv(T.TAB / "tipping_points.csv")
pert.to_csv(T.TAB / "threshold_perturbation.csv", index=False)
print("saved tipping_points.csv", out.shape, "and threshold_perturbation.csv", pert.shape, "to", T.TAB)

## Summary

Tipping points exist for every country and can be computed directly: each use case flips at percentile 100 × (1 − lowest pass rate among its requirements). Three numbers per country, the percentiles at which it enters *Partial*, *Minimum quality* and *High quality*, fully describe how its class depends on the percentile.

The percentile choice dominates. At P25 almost every country (198 of 201) meets no use case's minimum. Between P50 and P95, 88% of countries change class: at P50 nearly two thirds meet no use case's minimum (every low-income country and nearly every lower-middle-income one), while at P95 about 10% fail to. At P95, about three quarters of countries are also within five percentile points of a boundary, so classifications at the default are fragile to the exact percentile chosen.

Threshold values matter too. A ±10% change in all thresholds reclassifies 6–9% of countries, and ±25% up to almost a quarter. The library's gaming latency threshold (10 ms) alone moves almost a quarter of countries down a class at P95. Through video conferencing, upload capacity and latency decide most tipping points, not download speed.

Policy uses that need a classification (eligibility, prioritization) should therefore (i) state the percentile and justify it against the decision being made, (ii) flag countries within ±5 points of a boundary, and (iii) not let a single requirement, often NDT packet loss, decide a classification without corroboration.